# 02. Baseline YOLO

YOLO11s with Ultralytics defaults: the reference every later model has to beat.

**Validation protocol (shared by all notebooks, `src/core`).** Train and val clips are pooled (63 clips,
55 source flights) and split into 4 folds with `GroupKFold`, grouped by source flight, so no flight is in two folds.
For every fold a model is trained on the remaining folds and predicts every frame of the held-out one;
the out-of-fold (OOF) predictions cover each labelled frame once and are evaluated together.
The model kept per fold is `last.pt`: no epoch is picked by looking at the held-out fold.

If a Kaggle session ends mid-way, attach this notebook's output as input and rerun: finished folds are reused.

## Setup

In [ ]:
import subprocess
import sys
from pathlib import Path

# shared code (src/core) from the repo
if Path("../src/core").exists():
    sys.path.insert(0, "../src")
else:
    repo = Path("/tmp/repo")  # outside /kaggle/working so it is not saved as output
    if not repo.exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1",
                        "https://github.com/mohinska/ai_hw2_MultipleObjectVideoDetection", str(repo)], check=True)
    sys.path.insert(0, str(repo / "src"))

In [ ]:
!pip install -q ultralytics faster-coco-eval

In [ ]:
import pandas as pd

from core import CLASSES, N_FOLDS
from core.cv import cv_table, environment, run_cv
from core.data import assign_folds, find_data_root, load_pool
from core.fusion import fold_ensemble
from core.submission import write_submission
from core.yolo import make_yolo_fns, plot_curves

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("../outputs")
WORK.mkdir(parents=True, exist_ok=True)
environment()  # package versions and GPU; also saved to cv_baseline.json

## Config

In [ ]:
CFG = dict(
    name="baseline",
    model="yolo11s.pt",
    imgsz=1280,
    epochs=20,
    batch=8,
    train_step=5,      # every 5th frame (same-track IoU at step 5 is 0.59, see 01_eda)
    val_step=10,       # only for the per-epoch curve; metrics use every held-out frame
    conf=0.001,
    iou=0.7,
    max_det=500,
)
CFG

## Data and folds

In [ ]:
DATA = find_data_root()
frames, gt = load_pool(DATA)
frames["fold"] = frames["video"].map(assign_folds(frames))
frames.groupby("fold").agg(clips=("video", "nunique"), frames=("uid", "size"))

## Cross-validation
Per fold: train on the remaining folds, predict the held-out fold, save OOF predictions and metrics.

In [ ]:
train_fn, predict_fn = make_yolo_fns(CFG, WORK)
result = run_cv(frames, gt, CFG, train_fn, predict_fn, WORK)

## CV results
Per-fold rows show how much the score depends on which flights are held out; the OOF row evaluates
all labelled frames at once and is the number models are compared by. Config and metrics are saved
to `cv_baseline.json`.

In [ ]:
display(cv_table(result).round(4))
pd.Series({c: result["oof"][f"AP_{c}"] for c in CLASSES}).round(4).to_frame("OOF AP per class")

## Training curves
Ultralytics' own per-epoch validation on every 10th held-out frame, without ignored-region handling,
so the values differ from the COCO numbers above; they show convergence, not the final score.

In [ ]:
plot_curves(CFG["name"], N_FOLDS, WORK);

## Submission: ensemble of the 4 fold models
Each fold model missed a quarter of the clips; together they have seen all of them. Every fold model predicts
the test set and the boxes are fused by Weighted Boxes Fusion (`core.fusion`). The ensemble cannot be scored
on OOF (each frame was held out from only one model), so its gain shows only on the leaderboard.

In [ ]:
test = pd.read_csv(DATA / "test_images.csv")
test_dets = fold_ensemble(CFG["name"], predict_fn, [str(DATA / f) for f in test["file_name"]], WORK)
write_submission(test_dets, test, WORK / f"submission_{CFG['name']}.csv").head()